# Guide d'utilisation du plugin de backtest factoriel

Ce notebook montre un flux volontairement court : choisir les variables → charger les données utiles → calculer une fois le benchmark → tester les signaux unitaires → tester les ajouts incrémentaux → comparer plusieurs recettes composites.

Chaque backtest peut retourner une figure Plotly. Lors de l'export centralisé, les sorties HTML et PNG peuvent être activées séparément. Les figures présentent Top, Worst et Benchmark, ainsi que les ratios Top/Bench, Worst/Bench et Top/Worst.

## Points d'entrée principaux

- `test_unitary_signals()` : teste séparément level, pct, diff et rank_diff. Une simple liste récupère automatiquement la direction dans le catalogue `LOWER_IS_BETTER`; un dictionnaire peut remplacer cette direction et ajouter un dénominateur.
- `test_incremental_signals()` : ajoute chaque variable candidate séparément à un score de base. Les dimensions et les poids réellement utilisés proviennent de `candidate_config`.
- `test_composite_signals()` : exécute en une fois plusieurs recettes composites nommées et indépendantes.

Ces trois fonctions retournent toujours `{'screen': ..., 'results': ...}`. `calculate_composite_score()` conserve les colonnes dérivées dans `screen`.

## 1. Charger le plugin

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\Users\jingx\Downloads\公司回测插件")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import BacktestEngine
import func
import factor_config

importlib.reload(BacktestEngine)
importlib.reload(func)
importlib.reload(factor_config)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    build_periods_from_breakpoints,
    calculate_benchmark_performance,
    compare_backtest_results,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    prepare_performance_comparison,
    reconstruct_backtest_analysis,
    test_composite_signals,
    test_incremental_signals,
    test_unitary_signals,
)
from factor_config import (
    FACTOR_FAMILIES,
    RAW_VARIABLES as CATALOG_RAW_VARIABLES,
    factor_columns,
    signal_options,
)

print("Plugin chargé")

## 2. Choisir les familles et préparer la configuration

Le catalogue `FACTOR_FAMILIES` classe les colonnes du screen par famille. `factor_columns()` fournit une liste ciblée et `CATALOG_RAW_VARIABLES` contient tout le catalogue. Le benchmark et les paramètres communs du moteur sont définis une seule fois.

La sélection ci-dessous conserve le catalogue par famille, puis copie explicitement trois noms choisis après l'analyse unitaire. Cette petite liste est volontaire : elle rend les recettes composites faciles à lire et à modifier.

In [ ]:
DATA_DIR = PLUGIN_DIR / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
BENCHMARK = "STOXX EUROPE 600"
START_DATE = "2010-01-01"
PERIOD_BREAKPOINTS = list(RECOMMENDED_PERIOD_BREAKPOINTS)

SELECTED_FAMILIES = ("growth", "quality_profitability", "balance_sheet_leverage")
FAMILY_VARIABLES = factor_columns(*SELECTED_FAMILIES)
RAW_VARIABLES = [
    "Revenue 5Y CAGR",
    "FCF Conversion",
    "Net Debt to Ebit",
]
list_noire_path = None

In [ ]:
# Pour tout tester : RAW_VARIABLES = list(CATALOG_RAW_VARIABLES)
family_sizes = {name: len(columns) for name, columns in FACTOR_FAMILIES.items()}
display(pd.Series(family_sizes, name="Nombre de variables"))
display(pd.Series(RAW_VARIABLES, name="Variables sélectionnées"))

## 3. Charger les données et calculer une fois le benchmark

`load_backtest_data()` lit uniquement les variables demandées et les colonnes techniques. `calculate_benchmark_performance()` calcule ensuite le benchmark une seule fois. Sa série est injectée explicitement dans tous les tests par `RUN_OPTIONS`. Si `bench_perf` vaut `None`, le moteur calcule encore le benchmark lui-même.

In [ ]:
screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=RAW_VARIABLES,
    bench=BENCHMARK,
)
screen["Date"] = pd.to_datetime(screen["Date"])

print(f"screen : {screen.shape}, returns : {returns.shape}")

### Paramètres communs à tous les tests

Le benchmark est calculé avant les signaux. `RUN_OPTIONS` centralise ensuite le benchmark, la date de début, la fréquence de rebalancement, la méthode de remplissage, les périodes et les options de figure.

In [ ]:
BENCH_PERF = calculate_benchmark_performance(
    screen=screen, returns=returns, bench=BENCHMARK, start_date=START_DATE
)

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": 0.13,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": True,
    "build_figure": True,
}

In [ ]:
display(BENCH_PERF.rename("Benchmark").to_frame().tail())
display(pd.Series({key: value for key, value in RUN_OPTIONS.items() if key != "bench_perf"}))

## 4. Tester les signaux unitaires

Chaque variable est testée séparément en level, pct, diff et rank_diff. `rank_diff` calcule d'abord le rang orienté de la société dans sa région et son industrie, puis sa variation par rapport à la période précédente ; une valeur positive indique toujours une amélioration du rang. La liste simple récupère automatiquement `higher_is_better` dans le catalogue central, notamment `False` pour `Net Debt to Ebit`.

Pour ne tester que certaines dimensions, utilisez par exemple `dimensions=("rank_diff",)` ou `dimensions=("level", "diff")`.

In [ ]:
unitary_batch = test_unitary_signals(
    screen=screen,
    returns=returns,
    signal_config=RAW_VARIABLES,
    list_noire_path=list_noire_path,
    dimensions=("level", "pct", "diff", "rank_diff"),
    **RUN_OPTIONS,
)
screen = unitary_batch["screen"]
unitary_results = unitary_batch["results"]

In [ ]:
unitary_summary = pd.DataFrame([
    {
        "Signal": name,
        "Robust Score": result.get("robust_score"),
        "Top/Bench": result.get("top_bench_ratio"),
        "Top/Worst": result.get("top_worst_ratio"),
    }
    for name, result in unitary_results.items()
]).sort_values("Robust Score", ascending=False)
unitary_summary

Les colonnes pct, diff et rank_diff sont ajoutées au même `screen`. Elles restent disponibles pour une analyse manuelle ultérieure, sans relancer automatiquement un deuxième backtest identique.

In [ ]:
derived_columns = [
    column for column in screen.columns
    if ("__over__" in column or column.endswith("__pct")
        or column.endswith("__diff") or column.endswith("__rank_diff"))
]
derived_columns

## 5. Effectuer une analyse incrémentale

Le score de base est d'abord backtesté une fois. Chaque variable candidate est ensuite ajoutée séparément à cette base. Les dimensions level, pct, diff et rank_diff ainsi que leurs poids proviennent entièrement de `CANDIDATE_CONFIG`.

In [ ]:
BASELINE_CONFIG = {
    "Revenue 5Y CAGR": signal_options(level=1.0, pct=0.5),
}
CANDIDATE_CONFIG = {
    "FCF Conversion": signal_options(level=1.0, diff=0.5),
    "Net Debt to Ebit": signal_options(
        higher_is_better=False, level=1.0, diff=0.5
    ),
}

incremental_batch = test_incremental_signals(
    screen=screen,
    returns=returns,
    baseline_config=BASELINE_CONFIG,
    candidate_config=CANDIDATE_CONFIG,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = incremental_batch["screen"]
incremental_results = incremental_batch["results"]

In [ ]:
incremental_summary = pd.DataFrame([
    {
        "Test": name,
        "Robust Score": payload.get("robust_score"),
        "Top/Bench": payload.get("top_bench_ratio"),
        "Top/Worst": payload.get("top_worst_ratio"),
    }
    for name, payload in incremental_results.items()
]).sort_values("Robust Score", ascending=False)
incremental_summary

## 6. Construire et backtester plusieurs scores composites

Après les tests unitaires, définissez chaque recette avec trois ou quatre variables retenues. `signal_options()` évite de répéter les poids nuls : utilisez simplement `level`, `pct`, `diff`, `rank_diff`, `higher_is_better` et, si nécessaire, `denominator`. `test_composite_signals()` crée une colonne de score par recette, les ajoute au même `screen` et conserve dans chaque résultat le backtest et sa composition réellement utilisée.

In [ ]:
COMPOSITE_CONFIGS = {
    "Croissance mixte": {
        "Revenue 5Y CAGR": signal_options(level=1.0, pct=0.5),
        "FCF Conversion": signal_options(level=1.0, diff=0.5),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=1.0, diff=0.5
        ),
    },
    "Croissance variations": {
        "Revenue 5Y CAGR": signal_options(pct=1.0),
        "FCF Conversion": signal_options(diff=1.0),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=0.5, rank_diff=0.5
        ),
    },
}

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
composite_results = composite_batch["results"]

In [ ]:
composite_summary = pd.DataFrame([
    {
        "Combinaison": name,
        "Robust Score": payload.get("robust_score"),
        "Top/Bench": payload.get("top_bench_ratio"),
        "Top/Worst": payload.get("top_worst_ratio"),
    }
    for name, payload in composite_results.items()
]).sort_values("Robust Score", ascending=False)
display(composite_summary)

for name, payload in composite_results.items():
    print(f"Composition : {name}")
    display(payload["composition"])

## 7. Comparer et exporter tous les résultats

Placez les différents types de résultats dans un même dictionnaire. `compare_backtest_results()` produit une table triable qui présente également les variables brutes, les dimensions, les directions et la recette des poids de chaque test.

Il est conseillé de comparer d'abord `robust_rank_within_type` au sein d'une même famille de tests, puis d'examiner les composantes du Robust Score et enfin les courbes Top/Bench et Top/Worst. Pour les candidats incrémentaux, les colonnes `*_delta_vs_baseline` indiquent directement l'amélioration par rapport à la base.

In [ ]:
ALL_RESULTS = {
    "unitary": unitary_batch,
    "incremental": incremental_batch,
    "composite": composite_batch,
}

comparison_table = compare_backtest_results(ALL_RESULTS)
comparison_table

In [ ]:
classic_view = comparison_table[[
    "test_type", "test_name", "robust_score",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_information_ratio",
    "top_beta", "top_tracking_error", "top_sortino_ratio",
]].sort_values("robust_score", ascending=False)
classic_view

Le dossier d'export contient :

- `backtest_summary.csv` : toutes les métriques, les classements, les variables brutes, le résumé des poids et la recette lisible ;
- `signal_composition.csv` : une ligne par variable et dimension, avec le poids total de la variable brute et sa part du poids absolu ;
- `classic_metrics.csv` : les métriques classiques de Top, Worst et Benchmark au format long ;
- `period_metrics.csv` : les CAGR, les métriques classiques et les classements de chaque test pour chaque période ;
- `metrics_by_period.csv` : la table unifiée test × période qui réunit la période totale et toutes les sous-périodes ;
- `backtest_registry.json` : le registre complet lisible par machine ;
- `figures/` : les figures HTML ou PNG selon `export_html` et `export_png` ;
- `data/` : les courbes de valeur et les ratios de chaque test ;
- `holdings/` : les positions historiques Top et Worst de chaque test.

Tous les fichiers de données et toutes les tables récapitulatives sont écrits avant les images. L'export PNG dépend de Kaleido. Si les images ne sont pas nécessaires, désactivez à la fois `export_html` et `export_png`.

In [ ]:
# Décommentez cette ligne une seule fois si l'export PNG est indisponible.
# %pip install --upgrade kaleido

In [ ]:
export_bundle = export_backtest_results(
    results=ALL_RESULTS,
    output_dir=PLUGIN_DIR / "exports",
    export_name="factor_research_run",
    export_html=False,
    export_png=False,
)

export_bundle["export_dir"]

### Reconstruire et combiner les performances

`combine_backtest_performances()` utilise en priorité `ALL_RESULTS` en mémoire et complète les résultats absents avec les fichiers `data/*_performance.csv` du dossier exporté. Après un redémarrage du kernel, `globals().get("ALL_RESULTS")` renvoie `None` et la fonction restaure automatiquement toutes les performances depuis le disque.

`reconstruct_backtest_analysis()` restaure en une seule fois les performances, les compositions, tous les scores et métriques de la période totale, la table longue des métriques classiques et toutes les métriques par sous-période. `metrics_by_period` réunit `Période totale` et chaque période de rupture dans une seule table test × période, adaptée aux filtres et tableaux croisés. Après un redémarrage du kernel, les tables CSV officielles sont rechargées automatiquement ; si les résultats existent en mémoire, elles sont reconstruites directement. Le libellé automatique d'un composite présente ses variables brutes, les dimensions activées et leurs poids, par exemple `composite / Revenue 5Y CAGR[level×1.0, pct×0.5] | FCF Conversion[level×1.0, diff×0.5] | Net Debt to Ebit[level×1.0, diff×0.5] | Top`. `prepare_performance_comparison()` découvre automatiquement tous les tests du dossier exporté, construit `PERFORMANCE_SELECTION`, construit les ratios Top/Benchmark, Worst/Benchmark et Top/Worst, puis restitue les données sans tracer de figure.

In [ ]:
EXPORT_DIR = PLUGIN_DIR / "exports" / "demo_real_data_20260719"

analysis_bundle = reconstruct_backtest_analysis(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    portfolios=("Top",),
    performance_save_path=EXPORT_DIR / "all_top_performance.csv",
)

all_top_performance = analysis_bundle["performance"]
performance_composition = analysis_bundle["performance_composition"]
backtest_summary = analysis_bundle["summary"]
classic_metrics = analysis_bundle["classic_metrics"]
period_metrics = analysis_bundle["period_metrics"]
metrics_by_period = analysis_bundle["metrics_by_period"]
signal_composition = analysis_bundle["signal_composition"]

COMPOSITION_COLUMNS = [
    "display_path", "raw_variable", "dimension",
    "higher_is_better", "source_higher_is_better",
    "weight", "denominator", "raw_variable_total_weight",
    "raw_variable_absolute_weight_share",
]
available_composition_columns = [
    column for column in COMPOSITION_COLUMNS
    if column in performance_composition.columns
]
composite_composition = performance_composition.loc[
    performance_composition["test_type"].eq("composite"),
    available_composition_columns,
]

display(all_top_performance)
display(composite_composition)
display(backtest_summary)
display(classic_metrics)
display(metrics_by_period)

In [ ]:
comparison_data = prepare_performance_comparison(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    save_path=EXPORT_DIR / "selected_performance_comparison.csv",
)

selected_performance = comparison_data["performance"]
selected_ratios = comparison_data["ratios"]
performance_composition = comparison_data["composition"]
PERFORMANCE_SELECTION = comparison_data["performance_selection"]
RATIO_DEFINITIONS = comparison_data["ratio_definitions"]

comparison_figure = plot_performance_comparison(
    performance=selected_performance,
    ratios=selected_ratios,
    benchmark_column="Benchmark",
    title="Comparaison des performances",
    save_path=None,
    show_plot=True,
    rebase=True,
)

## 8. Comparer les différentes périodes

Tous les points d'entrée utilisent un seul paramètre temporel : la liste d'années `period_breakpoints`. La valeur recommandée `[2020, 2022, 2024]` découpe la chronologie en quatre segments : avant 2020, 2020–2021, 2022–2023 et depuis 2024. Le menu déroulant situé en haut de chaque figure Plotly affiche, pour la période choisie, le CAGR de Top, le CAGR actif et le CAGR Top/Worst. Les courbes Top, Worst et Benchmark repartent toujours de 100 au début de la période sélectionnée.

`backtest_summary.csv` contient les principaux champs au format large `period_<période>_<métrique>`. `period_metrics.csv` contient une ligne complète par couple test × période et convient mieux aux filtres et aux tableaux croisés.

In [ ]:
pd.DataFrame(build_periods_from_breakpoints(RUN_OPTIONS["period_breakpoints"]))

In [ ]:
period_table = period_metrics
period_view = period_table[[
    "period_label", "test_type", "test_name",
    "top_cagr", "active_cagr", "top_worst_cagr",
    "top_sharpe_ratio", "top_max_drawdown",
    "active_cagr_rank_global", "active_cagr_rank_within_type",
]]
period_view.sort_values(["period_label", "active_cagr_rank_global"])

In [ ]:
active_cagr_pivot = period_table.pivot_table(
    index=["test_type", "test_name"],
    columns="period_id",
    values="active_cagr",
)
active_cagr_pivot.sort_values(active_cagr_pivot.columns[-1], ascending=False)

Une année de rupture marque le début d'une nouvelle période. Ainsi, `[2009, 2020]` crée trois segments : avant 2009, 2009–2019 et depuis 2020. Les ruptures recommandées dans le code sont 2020, 2022 et 2024. Ajoutez 2009 si les données couvrent aussi les années antérieures à 2009. Une liste vide `[]` conserve uniquement la période totale.

In [ ]:
ALTERNATIVE_BREAKPOINTS = [2009, 2020]
display(pd.DataFrame(build_periods_from_breakpoints(ALTERNATIVE_BREAKPOINTS)))
# Modifiez RUN_OPTIONS["period_breakpoints"] avant les tests pour utiliser cette variante.

## 9. Lire les objets de résultats

Un résultat de backtest standard contient notamment :

```python
result["robust_score"]
result["top_bench_ratio"]
result["top_worst_ratio"]
result["performance"]
result["ratios"]
result["figure"]
result["top_holdings"]
result["worst_holdings"]
result["period_metrics"]
```

Les trois points d'entrée utilisent la même structure. Par exemple :

```python
unitary_batch["results"]["Net Debt to Ebit | level"]["robust_score"]
incremental_batch["results"]["Net Debt to Ebit"]["robust_score"]
composite_batch["results"]["Croissance mixte"]["robust_score"]
```

Le Robust Score nécessite au moins 756 jours de cotation communs. Si l'historique est insuffisant, il renvoie `NaN`.

Le flux interne est maintenant découplé : `calculate_top_vs_bottom_results()` produit d'abord les performances, les ratios et toutes les métriques, puis `plot_top_vs_bottom_results()` construit la figure à partir de cet objet. Pour un traitement en lot limité aux données, utilisez `show_plot=False, build_figure=False` dans n'importe quel point d'entrée ; les comparaisons restent reconstructibles depuis les fichiers performance locaux.